# Dropout Prediction — v3 (SOTA-Inspired)

Improvements over v2, grounded in 2024–2026 literature:

| Technique | Applied in |
|---|---|
| **SMOTE** class-imbalance oversampling | All stages |
| **Variance Threshold → Mutual Info → Boruta** feature selection | Stages 1, 3 |
| **XGBoost** with scale_pos_weight | Stages 1, 3 |
| **LSTM + Multi-Head Attention** (PyTorch) for sequential sessions | Stage 2 |
| **DeepSurv-inspired deep MLP** (PyTorch) | Stage 4 |
| **F1\*** calibration metric: `(macro-F1 + accuracy) / 2` | Evaluation |
| CV-optimal threshold (maximise macro-F1) | All stages |

In [1]:
# Install extra dependencies (skip if already installed)
!pip install -q boruta imbalanced-learn xgboost


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report
)

import xgboost as xgb
from imblearn.over_sampling import SMOTE
from boruta import BorutaPy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

GRADE_MAP    = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
YEAR_MAP     = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}
RANDOM_STATE = 42
DEVICE       = torch.device('cpu')  # swap to 'cuda' if available

print('All imports OK.')

All imports OK.


In [3]:
# ── Shared encoding helpers ───────────────────────────────────────────────────

def _base_encode(df):
    df = df.copy()
    df['external_flag'] = df['External'].map({'Y': 1, 'N': 0})
    df['year_num']      = df['Year'].str.lower().map(YEAR_MAP)
    return df

def _grade_cols(df, cols):
    for col in cols:
        df[col.replace(' ', '_') + '_num'] = df[col].str.upper().map(GRADE_MAP)
    return df

def _shared_features(df, sess_cols, grade_num_cols):
    """Compute engagement + session stats used in every stage."""
    df['session_total']   = df[sess_cols].fillna(0).sum(axis=1)
    df['session_mean']    = df[sess_cols].mean(axis=1)
    df['session_std']     = df[sess_cols].std(axis=1).fillna(0)
    df['session_trend']   = df[sess_cols[-1]].fillna(0) - df[sess_cols[0]].fillna(0)
    df['engagement']      = df['forum Q'].fillna(0) + df['forum A'].fillna(0)
    df['eng_per_session'] = df['engagement'] / (df['session_total'] + 1)
    df['oh_norm']         = df['office hour visits'].fillna(0)
    if len(grade_num_cols) >= 2:
        df['grade_mean']  = df[grade_num_cols].mean(axis=1)
        df['grade_delta'] = df[grade_num_cols[-1]] - df[grade_num_cols[0]]
        df['at_risk']     = (df['grade_mean'] < 2).astype(int)
        df['high_perf']   = (df['grade_mean'] >= 4).astype(int)
    if len(grade_num_cols) >= 3:
        df['grade_accel'] = (
            (df[grade_num_cols[-1]] - df[grade_num_cols[-2]]) -
            (df[grade_num_cols[-2]] - df[grade_num_cols[-3]])
        )
    df['yr_grade_ix'] = df['year_num'] * df.get('grade_mean', df[grade_num_cols[0]])
    return df

# ── Optimal threshold search ──────────────────────────────────────────────────

def _optimal_threshold(y_true, probas):
    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(0.1, 0.9, 81):
        preds = (probas >= t).astype(int)
        f = f1_score(y_true, preds, average='macro', zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t

# ── F1* calibration metric ────────────────────────────────────────────────────

def f1_star(y_true, y_pred):
    """Arithmetic mean of macro-F1 and accuracy (Calibrating F1, 2026)."""
    return (f1_score(y_true, y_pred, average='macro') + accuracy_score(y_true, y_pred)) / 2

print('Shared helpers loaded.')

Shared helpers loaded.


In [4]:
# ── Multi-stage feature selection ─────────────────────────────────────────────
# Variance Threshold → Mutual Information (top-k) → Boruta
# (Multi-Stage Feature Selection for Student Dropout Prediction, 2025)

def select_features(X_imp, y, feature_names, mi_top=None, verbose=False):
    """Returns boolean mask over feature_names."""
    names = np.array(feature_names)
    mask  = np.ones(len(names), dtype=bool)

    # Stage 1: Variance Threshold
    vt = VarianceThreshold(threshold=0.01)
    vt.fit(X_imp)
    mask &= vt.get_support()
    if verbose:
        print(f'  After VT: {mask.sum()} / {len(mask)} features')

    # Stage 2: Mutual Information — keep top-k (default: top 80%)
    X_vt = X_imp[:, mask]
    mi   = mutual_info_classif(X_vt, y, random_state=RANDOM_STATE)
    k    = mi_top or max(1, int(0.80 * mask.sum()))
    mi_keep = np.zeros(mask.sum(), dtype=bool)
    mi_keep[np.argsort(mi)[-k:]] = True
    tmp = np.zeros(len(mask), dtype=bool)
    tmp[np.where(mask)[0][mi_keep]] = True
    mask = tmp
    if verbose:
        print(f'  After MI: {mask.sum()} features')

    # Stage 3: Boruta
    rf = RandomForestClassifier(
        n_estimators=100, max_depth=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    )
    boruta = BorutaPy(rf, n_estimators='auto', max_iter=50,
                      random_state=RANDOM_STATE, verbose=0)
    boruta.fit(X_imp[:, mask], y)
    boruta_mask = boruta.support_ | boruta.support_weak_
    tmp2 = np.zeros(len(mask), dtype=bool)
    tmp2[np.where(mask)[0][boruta_mask]] = True
    mask = tmp2
    if verbose:
        print(f'  After Boruta: {mask.sum()} features')
        print(f'  Selected: {list(names[mask])}')

    # Fallback: never drop below 5 features
    if mask.sum() < 5:
        mi_full = mutual_info_classif(X_imp, y, random_state=RANDOM_STATE)
        mask    = np.zeros(len(names), dtype=bool)
        mask[np.argsort(mi_full)[-10:]] = True
    return mask

print('Feature selection ready.')

Feature selection ready.


In [5]:
# ── PyTorch models ────────────────────────────────────────────────────────────

class LSTMAttention(nn.Module):
    """LSTM + Multi-Head Attention for sequential session data.
    Inspired by Zhou et al. (2026) — 92.4% accuracy, 0.90 F1."""

    def __init__(self, seq_feat, static_feat,
                 hidden=64, num_layers=2, num_heads=4, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(seq_feat, hidden, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.attn = nn.MultiheadAttention(hidden, num_heads,
                                          dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(hidden)
        self.head = nn.Sequential(
            nn.Linear(hidden + static_feat, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),          # logit
        )

    def forward(self, seq, static):
        out, _      = self.lstm(seq)            # (B, T, H)
        attn_out, _ = self.attn(out, out, out)  # (B, T, H)
        pooled      = self.norm(attn_out).mean(dim=1)  # (B, H)
        return self.head(torch.cat([pooled, static], dim=1)).squeeze(-1)


class DeepSurvNet(nn.Module):
    """Deep MLP inspired by DeepSurv (Danesi, 2024/2026).
    Adapted for binary classification with Cox-like deep architecture."""

    def __init__(self, in_features, hidden=(256, 128, 64), dropout=0.3):
        super().__init__()
        layers, prev = [], in_features
        for h in hidden:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers.append(nn.Linear(prev, 1))  # logit
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ── Training / inference helpers ──────────────────────────────────────────────

def _pos_weight(y):
    neg, pos = (y == 0).sum(), (y == 1).sum()
    return torch.tensor([neg / max(pos, 1)], dtype=torch.float32)


def train_flat(model, X, y, epochs=80, lr=1e-3, batch=256):
    """Train a model that accepts a single flat tensor X."""
    ds     = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y.astype(float)))
    loader = DataLoader(ds, batch_size=batch, shuffle=True)
    crit   = nn.BCEWithLogitsLoss(pos_weight=_pos_weight(y))
    opt    = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            opt.zero_grad()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            crit(model(xb), yb).backward()
            opt.step()
        sched.step()
    return model


def train_lstm(model, X_seq, X_sta, y, epochs=80, lr=1e-3, batch=256):
    """Train LSTMAttention model."""
    ds = TensorDataset(
        torch.FloatTensor(X_seq),
        torch.FloatTensor(X_sta),
        torch.FloatTensor(y.astype(float)),
    )
    loader = DataLoader(ds, batch_size=batch, shuffle=True)
    crit   = nn.BCEWithLogitsLoss(pos_weight=_pos_weight(y))
    opt    = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    model.train()
    for _ in range(epochs):
        for seq_b, sta_b, yb in loader:
            opt.zero_grad()
            loss = crit(model(seq_b, sta_b), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
    return model


@torch.no_grad()
def infer_flat(model, X):
    model.eval()
    return torch.sigmoid(model(torch.FloatTensor(X))).numpy()


@torch.no_grad()
def infer_lstm(model, X_seq, X_sta):
    model.eval()
    return torch.sigmoid(
        model(torch.FloatTensor(X_seq), torch.FloatTensor(X_sta))
    ).numpy()


print('PyTorch models defined.')

PyTorch models defined.


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Stage 1 — XGBoost + SMOTE + Boruta  (after session 2 / test 1)
# ─────────────────────────────────────────────────────────────────────────────

def predict_dropout1():
    SESS = ['session 1', 'session 2']
    GNUM = ['test_1_num']

    train = _shared_features(
        _grade_cols(_base_encode(pd.read_csv('ND26_dropout.csv')), ['test 1']),
        SESS, GNUM
    )
    train['grade_eng_ix'] = train['test_1_num'] * train['engagement']
    train['y'] = train['dropout'].map({'Y': 1, 'N': 0})

    feats = [
        'external_flag', 'year_num',
        'session 1', 'session 2', 'session_mean', 'session_total', 'session_std',
        'test_1_num', 'forum Q', 'forum A', 'engagement', 'eng_per_session',
        'oh_norm', 'yr_grade_ix', 'grade_eng_ix',
    ]

    imp   = SimpleImputer(strategy='median')
    X_imp = imp.fit_transform(train[feats])
    y_tr  = train['y'].values

    # Multi-stage feature selection
    mask  = select_features(X_imp, y_tr, feats, verbose=True)
    feats_sel = [f for f, m in zip(feats, mask) if m]
    X_sel = X_imp[:, mask]

    # SMOTE
    sm = SMOTE(random_state=RANDOM_STATE)
    X_res, y_res = sm.fit_resample(X_sel, y_tr)

    neg, pos = (y_res == 0).sum(), (y_res == 1).sum()
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=neg / pos,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    model.fit(X_res, y_res)

    test = _shared_features(
        _grade_cols(_base_encode(pd.read_csv('./data/entry_dropout1.csv')), ['test 1']),
        SESS, GNUM
    )
    test['grade_eng_ix'] = test['test_1_num'] * test['engagement']

    X_te   = imp.transform(test[feats])[:, mask]
    probas = model.predict_proba(X_te)[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Stage 2 — LSTM+Attention ⊕ XGBoost ensemble + SMOTE
#           (after session 4 / test 2)
# (Zhou et al., 2026 — Multi-Head Attention LSTM)
# ─────────────────────────────────────────────────────────────────────────────

def _build_lstm_arrays_s2(df, imp_sess, imp_sta, fit=False):
    """Return X_seq (N, 4, 1), X_sta (N, 8) arrays."""
    sess_cols = ['session 1', 'session 2', 'session 3', 'session 4']
    sta_cols  = [
        'external_flag', 'year_num', 'test_1_num', 'test_2_num',
        'grade_mean', 'grade_delta', 'at_risk', 'engagement',
    ]
    if fit:
        X_seq_raw = imp_sess.fit_transform(df[sess_cols])
        X_sta_raw = imp_sta.fit_transform(df[sta_cols])
    else:
        X_seq_raw = imp_sess.transform(df[sess_cols])
        X_sta_raw = imp_sta.transform(df[sta_cols])
    X_seq = X_seq_raw.reshape(-1, 4, 1).astype(np.float32)
    return X_seq, X_sta_raw.astype(np.float32)


def predict_dropout2():
    SESS = ['session 1', 'session 2', 'session 3', 'session 4']
    GNUM = ['test_1_num', 'test_2_num']

    train = _shared_features(
        _grade_cols(_base_encode(pd.read_csv('ND26_dropout.csv')), ['test 1', 'test 2']),
        SESS, GNUM
    )
    train['y'] = train['dropout'].map({'Y': 1, 'N': 0})
    train = train[train['session 3'].notna() | train['test 2'].notna()]

    imp_sess = SimpleImputer(strategy='median')
    imp_sta  = SimpleImputer(strategy='median')
    X_seq, X_sta = _build_lstm_arrays_s2(train, imp_sess, imp_sta, fit=True)
    y_tr = train['y'].values

    # SMOTE on flat representation for XGB branch
    X_flat = np.concatenate([X_seq.reshape(len(X_seq), -1), X_sta], axis=1)
    sm = SMOTE(random_state=RANDOM_STATE)
    X_res, y_res = sm.fit_resample(X_flat, y_tr)

    # ── Branch A: LSTM + Attention ────────────────────────────────────────────
    scaler_seq = StandardScaler()
    scaler_sta = StandardScaler()
    X_seq_s = scaler_seq.fit_transform(X_seq.reshape(len(X_seq), -1)).reshape(-1, 4, 1)
    X_sta_s = scaler_sta.fit_transform(X_sta)

    # SMOTE on scaled
    X_flat_s = np.concatenate([X_seq_s.reshape(len(X_seq_s), -1), X_sta_s], axis=1)
    X_res_s, y_res_s = sm.fit_resample(X_flat_s, y_tr)
    X_seq_res = X_res_s[:, :4].reshape(-1, 4, 1)
    X_sta_res = X_res_s[:, 4:]

    lstm = LSTMAttention(seq_feat=1, static_feat=8, hidden=64, num_heads=4)
    lstm = train_lstm(lstm, X_seq_res, X_sta_res, y_res_s, epochs=80)

    # ── Branch B: XGBoost ─────────────────────────────────────────────────────
    neg, pos = (y_res == 0).sum(), (y_res == 1).sum()
    xgb_m = xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=neg / pos,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    xgb_m.fit(X_res, y_res)

    # ── Inference ─────────────────────────────────────────────────────────────
    test = _shared_features(
        _grade_cols(_base_encode(pd.read_csv('./data/entry_dropout2.csv')), ['test 1', 'test 2']),
        SESS, GNUM
    )
    X_seq_te, X_sta_te = _build_lstm_arrays_s2(test, imp_sess, imp_sta, fit=False)
    X_flat_te = np.concatenate([X_seq_te.reshape(len(X_seq_te), -1), X_sta_te], axis=1)

    X_seq_te_s = scaler_seq.transform(X_seq_te.reshape(len(X_seq_te), -1)).reshape(-1, 4, 1)
    X_sta_te_s = scaler_sta.transform(X_sta_te)

    p_lstm = infer_lstm(lstm, X_seq_te_s, X_sta_te_s)
    p_xgb  = xgb_m.predict_proba(X_flat_te)[:, 1]
    probas = 0.5 * p_lstm + 0.5 * p_xgb     # soft ensemble

    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Stage 3 — Boruta + XGBoost + SMOTE  (after session 5 / test 3)
# (Multi-Stage Feature Selection 2025 + SMOTE-GAN 2026)
# ─────────────────────────────────────────────────────────────────────────────

def predict_dropout3():
    SESS = ['session 1', 'session 2', 'session 3', 'session 4', 'session 5']
    GNUM = ['test_1_num', 'test_2_num', 'test_3_num']

    train = _shared_features(
        _grade_cols(
            _base_encode(pd.read_csv('ND26_dropout.csv')),
            ['test 1', 'test 2', 'test 3']
        ),
        SESS, GNUM
    )
    train['y'] = train['dropout'].map({'Y': 1, 'N': 0})
    train = train[train['session 5'].notna() | train['test 3'].notna()]

    feats = [
        'external_flag', 'year_num',
        'session 1', 'session 2', 'session 3', 'session 4', 'session 5',
        'session_mean', 'session_std', 'session_total', 'session_trend',
        'test_1_num', 'test_2_num', 'test_3_num',
        'grade_mean', 'grade_delta', 'grade_accel', 'at_risk', 'high_perf',
        'forum Q', 'forum A', 'engagement', 'eng_per_session',
        'oh_norm', 'yr_grade_ix',
    ]

    imp   = SimpleImputer(strategy='median')
    X_imp = imp.fit_transform(train[feats])
    y_tr  = train['y'].values

    mask     = select_features(X_imp, y_tr, feats, verbose=True)
    X_sel    = X_imp[:, mask]

    sm = SMOTE(random_state=RANDOM_STATE)
    X_res, y_res = sm.fit_resample(X_sel, y_tr)

    neg, pos = (y_res == 0).sum(), (y_res == 1).sum()
    model = xgb.XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.04,
        subsample=0.8, colsample_bytree=0.8, gamma=1,
        scale_pos_weight=neg / pos,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    model.fit(X_res, y_res)

    test = _shared_features(
        _grade_cols(
            _base_encode(pd.read_csv('./data/entry_dropout3.csv')),
            ['test 1', 'test 2', 'test 3']
        ),
        SESS, GNUM
    )

    X_te   = imp.transform(test[feats])[:, mask]
    probas = model.predict_proba(X_te)[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Stage 4 — DeepSurvNet ⊕ XGBoost ⊕ HGB soft ensemble + SMOTE
#           (end-of-year, all features)
# (Danesi 2024/2026 DeepSurv; SMOTE-GAN 2026)
# ─────────────────────────────────────────────────────────────────────────────

def predict_dropout4():
    SESS = ['session 1','session 2','session 3','session 4','session 5','session 6']
    GCOLS = ['test 1','test 2','test 3','ind cw','group cw','final grade']
    GNUM  = ['test_1_num','test_2_num','test_3_num',
             'ind_cw_num','group_cw_num','final_grade_num']

    train = _shared_features(
        _grade_cols(_base_encode(pd.read_csv('ND26_dropout.csv')), GCOLS),
        SESS, GNUM
    )
    num_tests = ['test_1_num','test_2_num','test_3_num']
    num_cw    = ['ind_cw_num','group_cw_num']
    train['test_mean']     = train[num_tests].mean(axis=1)
    train['cw_mean']       = train[num_cw].mean(axis=1)
    train['cw_vs_test']    = train['cw_mean'] - train['test_mean']
    train['final_vs_init'] = train['final_grade_num'] - train['test_1_num']
    train['y'] = train['dropout'].map({'Y': 1, 'N': 0})
    train = train[train['session 6'].notna() | train['ind cw'].notna()]

    feats = [
        'external_flag', 'year_num',
        'session 1','session 2','session 3','session 4','session 5','session 6',
        'session_mean','session_std','session_total','session_trend',
        'test_1_num','test_2_num','test_3_num','ind_cw_num','group_cw_num','final_grade_num',
        'test_mean','cw_mean','grade_mean','grade_delta','grade_accel',
        'cw_vs_test','final_vs_init','at_risk','high_perf',
        'forum Q','forum A','engagement','eng_per_session',
        'oh_norm','yr_grade_ix',
    ]

    imp    = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    X_imp  = imp.fit_transform(train[feats])
    X_sc   = scaler.fit_transform(X_imp)
    y_tr   = train['y'].values

    # SMOTE on imputed (unscaled) for XGB/HGB
    sm = SMOTE(random_state=RANDOM_STATE)
    X_res, y_res = sm.fit_resample(X_imp, y_tr)

    # SMOTE on scaled for DeepSurv
    X_sc_res, y_sc_res = sm.fit_resample(X_sc, y_tr)

    # ── DeepSurv deep MLP ─────────────────────────────────────────────────────
    deep = DeepSurvNet(in_features=X_sc.shape[1], hidden=(256, 128, 64), dropout=0.3)
    deep = train_flat(deep, X_sc_res, y_sc_res, epochs=100, lr=1e-3)

    # ── XGBoost ───────────────────────────────────────────────────────────────
    neg, pos = (y_res == 0).sum(), (y_res == 1).sum()
    xgb_m = xgb.XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.04,
        subsample=0.8, colsample_bytree=0.8, gamma=1,
        scale_pos_weight=neg / pos,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    xgb_m.fit(X_res, y_res)

    # ── HistGradientBoosting ──────────────────────────────────────────────────
    hgb = HistGradientBoostingClassifier(
        max_iter=300, max_depth=6, learning_rate=0.05,
        min_samples_leaf=20, l2_regularization=0.1,
        class_weight='balanced', random_state=RANDOM_STATE,
    )
    hgb.fit(X_res, y_res)

    # ── Inference ─────────────────────────────────────────────────────────────
    test = _shared_features(
        _grade_cols(_base_encode(pd.read_csv('./data/entry_dropout4.csv')), GCOLS),
        SESS, GNUM
    )
    test['test_mean']     = test[num_tests].mean(axis=1)
    test['cw_mean']       = test[num_cw].mean(axis=1)
    test['cw_vs_test']    = test['cw_mean'] - test['test_mean']
    test['final_vs_init'] = test['final_grade_num'] - test['test_1_num']

    X_te_imp = imp.transform(test[feats])
    X_te_sc  = scaler.transform(X_te_imp)

    p_deep = infer_flat(deep, X_te_sc)
    p_xgb  = xgb_m.predict_proba(X_te_imp)[:, 1]
    p_hgb  = hgb.predict_proba(X_te_imp)[:, 1]

    # Weighted soft ensemble: DeepSurv 40%, XGBoost 35%, HGB 25%
    probas = 0.40 * p_deep + 0.35 * p_xgb + 0.25 * p_hgb

    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [ ]:
print('Stage 1:', predict_dropout1())
print('Stage 2:', predict_dropout2())
print('Stage 3:', predict_dropout3())
print('Stage 4:', predict_dropout4())

  After VT: 15 / 15 features
  After MI: 12 features
  After Boruta: 12 features
  Selected: [np.str_('session 1'), np.str_('session 2'), np.str_('session_mean'), np.str_('session_total'), np.str_('test_1_num'), np.str_('forum Q'), np.str_('forum A'), np.str_('engagement'), np.str_('eng_per_session'), np.str_('oh_norm'), np.str_('yr_grade_ix'), np.str_('grade_eng_ix')]
Stage 1: ['Y', 'N', 'N', 'Y', 'N', 'Y', 'Y', 'N', 'N', 'Y']


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Cross-Validation Evaluation with F1* Calibration
# ═════════════════════════════════════════════════════════════════════════════
#
# F1* = (macro-F1 + accuracy) / 2
# (Calibrating F1 Scores for Fair Performance Comparison, 2026)
#
# NOTE: LSTM branch in Stage 2 is excluded from sklearn cross_val_predict
#       (PyTorch training loop is not a sklearn estimator).
#       Stage 2 evaluation uses XGBoost-only branch for CV compatibility.
# ═════════════════════════════════════════════════════════════════════════════

train_full = pd.read_csv('ND26_dropout.csv')
cv         = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def _eval_stage(name, X, y, pipe, cv):
    p_cv   = cross_val_predict(pipe, X, y, cv=cv, method='predict_proba')[:, 1]
    t_opt  = _optimal_threshold(y, p_cv)
    y_pred = (p_cv >= t_opt).astype(int)
    acc    = accuracy_score(y, y_pred)
    mf1    = f1_score(y, y_pred, average='macro')
    fs     = f1_star(y, y_pred)
    auc    = roc_auc_score(y, p_cv)
    dr     = f1_score(y, y_pred, average=None)[1]
    print(f'{'=' * 64}')
    print(f'{name}')
    print(f'  Threshold (CV-optimal): {t_opt:.2f}')
    print(f'  Accuracy : {acc:.4f}   ROC-AUC : {auc:.4f}')
    print(f'  Macro-F1 : {mf1:.4f}   F1*     : {fs:.4f}')
    print(f'  Dropout Recall: {dr:.4f}')
    print()
    print(classification_report(y, y_pred, target_names=['No Dropout (N)', 'Dropout (Y)']))
    return dict(stage=name, accuracy=round(acc,4), roc_auc=round(auc,4),
                macro_f1=round(mf1,4), f1_star=round(fs,4),
                dropout_recall=round(dr,4))


rows = []

# ── Stage 1 ──────────────────────────────────────────────────────────────────
SESS1 = ['session 1', 'session 2']
GNUM1 = ['test_1_num']
t1 = _shared_features(
    _grade_cols(_base_encode(train_full.copy()), ['test 1']),
    SESS1, GNUM1
)
t1['grade_eng_ix'] = t1['test_1_num'] * t1['engagement']
t1['y'] = t1['dropout'].map({'Y': 1, 'N': 0})
f1 = [
    'external_flag', 'year_num',
    'session 1', 'session 2', 'session_mean', 'session_total', 'session_std',
    'test_1_num', 'forum Q', 'forum A', 'engagement', 'eng_per_session',
    'oh_norm', 'yr_grade_ix', 'grade_eng_ix',
]
# Pre-select features on full training set, then fix for CV
imp1 = SimpleImputer(strategy='median')
X1_imp = imp1.fit_transform(t1[f1])
m1     = select_features(X1_imp, t1['y'].values, f1)
f1_sel = [f for f, keep in zip(f1, m1) if keep]
pipe1 = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('xgb', xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
])
rows.append(_eval_stage(
    'STAGE 1 — XGBoost + SMOTE + Boruta  (after session 2 / test 1)',
    t1[f1_sel], t1['y'].values, pipe1, cv
))

# ── Stage 2 (XGB branch only, CV-compatible) ──────────────────────────────
SESS2 = ['session 1','session 2','session 3','session 4']
GNUM2 = ['test_1_num','test_2_num']
t2 = _shared_features(
    _grade_cols(_base_encode(train_full.copy()), ['test 1','test 2']),
    SESS2, GNUM2
)
t2['y'] = t2['dropout'].map({'Y': 1, 'N': 0})
t2 = t2[t2['session 3'].notna() | t2['test 2'].notna()]
f2 = [
    'external_flag','year_num',
    'session 1','session 2','session 3','session 4',
    'session_mean','session_std','session_total',
    'test_1_num','test_2_num','grade_mean','grade_delta','at_risk',
    'forum Q','forum A','engagement','eng_per_session','oh_norm','yr_grade_ix',
]
pipe2 = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('xgb', xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
])
rows.append(_eval_stage(
    'STAGE 2 — LSTM+Attn ⊕ XGBoost + SMOTE  (after session 4 / test 2) [XGB branch CV]',
    t2[f2], t2['y'].values, pipe2, cv
))

# ── Stage 3 ──────────────────────────────────────────────────────────────────
SESS3 = ['session 1','session 2','session 3','session 4','session 5']
GNUM3 = ['test_1_num','test_2_num','test_3_num']
t3 = _shared_features(
    _grade_cols(_base_encode(train_full.copy()), ['test 1','test 2','test 3']),
    SESS3, GNUM3
)
t3['y'] = t3['dropout'].map({'Y': 1, 'N': 0})
t3 = t3[t3['session 5'].notna() | t3['test 3'].notna()]
f3 = [
    'external_flag','year_num',
    'session 1','session 2','session 3','session 4','session 5',
    'session_mean','session_std','session_total','session_trend',
    'test_1_num','test_2_num','test_3_num',
    'grade_mean','grade_delta','grade_accel','at_risk','high_perf',
    'forum Q','forum A','engagement','eng_per_session','oh_norm','yr_grade_ix',
]
imp3   = SimpleImputer(strategy='median')
X3_imp = imp3.fit_transform(t3[f3])
m3     = select_features(X3_imp, t3['y'].values, f3)
f3_sel = [f for f, keep in zip(f3, m3) if keep]
pipe3 = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('xgb', xgb.XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.04,
        subsample=0.8, colsample_bytree=0.8, gamma=1,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
])
rows.append(_eval_stage(
    'STAGE 3 — Boruta + XGBoost + SMOTE  (after session 5 / test 3)',
    t3[f3_sel], t3['y'].values, pipe3, cv
))

# ── Stage 4 ──────────────────────────────────────────────────────────────────
SESS4  = ['session 1','session 2','session 3','session 4','session 5','session 6']
GCOLS4 = ['test 1','test 2','test 3','ind cw','group cw','final grade']
GNUM4  = ['test_1_num','test_2_num','test_3_num','ind_cw_num','group_cw_num','final_grade_num']
t4 = _shared_features(
    _grade_cols(_base_encode(train_full.copy()), GCOLS4),
    SESS4, GNUM4
)
for nt, nc in [(['test_1_num','test_2_num','test_3_num'], ['ind_cw_num','group_cw_num'])]:
    t4['test_mean']     = t4[nt].mean(axis=1)
    t4['cw_mean']       = t4[nc].mean(axis=1)
    t4['cw_vs_test']    = t4['cw_mean'] - t4['test_mean']
    t4['final_vs_init'] = t4['final_grade_num'] - t4['test_1_num']
t4['y'] = t4['dropout'].map({'Y': 1, 'N': 0})
t4 = t4[t4['session 6'].notna() | t4['ind cw'].notna()]
f4 = [
    'external_flag','year_num',
    'session 1','session 2','session 3','session 4','session 5','session 6',
    'session_mean','session_std','session_total','session_trend',
    'test_1_num','test_2_num','test_3_num','ind_cw_num','group_cw_num','final_grade_num',
    'test_mean','cw_mean','grade_mean','grade_delta','grade_accel',
    'cw_vs_test','final_vs_init','at_risk','high_perf',
    'forum Q','forum A','engagement','eng_per_session','oh_norm','yr_grade_ix',
]
# DeepSurv branch not CV-compatible; use XGB+HGB ensemble for CV eval
pipe4 = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('xgb', xgb.XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.04,
        subsample=0.8, colsample_bytree=0.8, gamma=1,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
])
rows.append(_eval_stage(
    'STAGE 4 — DeepSurv ⊕ XGBoost ⊕ HGB + SMOTE  (end-of-year) [XGB branch CV]',
    t4[f4], t4['y'].values, pipe4, cv
))

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
summary = pd.DataFrame(rows).set_index('stage')

# v2 results for comparison
v2 = pd.DataFrame([
    dict(stage='S1', accuracy=0.9521, roc_auc=0.9911, macro_f1=0.9518, f1_star=(0.9518+0.9521)/2, dropout_recall=0.9478),
    dict(stage='S2', accuracy=0.9479, roc_auc=0.9881, macro_f1=0.9455, f1_star=(0.9455+0.9479)/2, dropout_recall=0.9341),
    dict(stage='S3', accuracy=0.9703, roc_auc=0.9949, macro_f1=0.9609, f1_star=(0.9609+0.9703)/2, dropout_recall=0.9417),
    dict(stage='S4', accuracy=0.9824, roc_auc=0.9962, macro_f1=0.9569, f1_star=(0.9569+0.9824)/2, dropout_recall=0.9238),
]).set_index('stage')

print('\n── v3 Model Summary ──')
print(summary[['accuracy','roc_auc','macro_f1','f1_star','dropout_recall']].to_string())

print('\n── v2 Baseline ──')
print(v2.to_string())

print('\n── Delta (v3 − v2) ──')
delta = summary[['accuracy','roc_auc','macro_f1','f1_star','dropout_recall']].values \
      - v2.values
print(pd.DataFrame(delta, columns=['accuracy','roc_auc','macro_f1','f1_star','dropout_recall'],
                   index=['S1','S2','S3','S4']).to_string())